# Integrating the KA1 mobility dataset with Cost of Living data (Eurostat HICP)

This notebook:
1. Loads `df_mobility` from `dataframe_1000_examples.csv`
2. Loads `df_hicp` from `prc_hicp_aind_linear_csv.gz` (Eurostat `prc_hicp_aind` dataset)
3. Adds two new columns to `df_mobility`:
   - `Sending Country HICP` (right after `Sending Country`)
   - `Receiving Country HICP` (right after `Receiving Country`)
4. The join is done on the country name and on the **Academic Year**, which in the HICP dataset corresponds to `TIME_PERIOD`.

**Note on the chosen measure**: the Eurostat `prc_hicp_aind` file contains several metrics (`unit`) and hundreds of expenditure sub-categories (`coicop`). As a proxy for the overall "cost of living" per country/year, I used the measure most representative of the general cost of living:
- `unit = "Annual average index"` (annual average consumer price index)
- `coicop = "All-items HICP"` (full basket, not a single expenditure category)

If you had a different unit/coicop combination in mind, you can change the two filters in cell 4; the following cell prints the available options.

In [2]:
import pandas as pd
import numpy as np

MOBILITY_PATH = "Datasets/Processed/Erasmus-Data.csv"
HICP_PATH = "Datasets/Raw/prc_hicp_aind_linear.csv.gz"

## 1. Loading df_mobility

In [ ]:
df_mobility = pd.read_csv(MOBILITY_PATH, index_col=0)
df_mobility.head()

C:\Users\rpasq\AppData\Local\Temp\ipykernel_15584\3755031370.py:1: DtypeWarning: Columns (0: Fewer Opportunities) have mixed types. Specify dtype option on import or set low_memory=False.
  df_mobility = pd.read_csv(MOBILITY_PATH)


,Unnamed: 0,Academic Year,Mobility Duration,Field of Education,Participant Country,Education Level,Participant Gender,Fewer Opportunities,Participant Age,Sending Country,Sending City,Sending Organization,Receiving Country,Receiving City,Receiving Organization
0,1358,2023,61,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,Austria,WIEN,WU,Germany,Munich,-
1,1359,2023,101,Economics,Germany,ISCED-7 - Second cycle / Master’s or equivalen...,Male,0,27,Austria,WIEN,WU,France,Toulouse,Université Toulouse Capitole
2,1360,2023,121,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,Austria,WIEN,WU,France,Paris,-
3,1361,2023,171,Business and administration,Germany,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,22,Austria,WIEN,WU,France,Paris,-
4,1365,2023,86,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,23,Austria,WIEN,WU,Germany,Hamburg,-


## 2. Loading df_hicp

The gz file already contains a formatted csv (columns: `DATAFLOW, LAST UPDATE, freq, unit, coicop, geo, TIME_PERIOD, OBS_VALUE, OBS_FLAG, CONF_STATUS`).

In [46]:
df_hicp = pd.read_csv(HICP_PATH, compression="gzip", low_memory=False)
df_hicp.head()

,DATAFLOW,LAST UPDATE,freq,unit,coicop,geo,TIME_PERIOD,OBS_VALUE,OBS_FLAG,CONF_STATUS
0,ESTAT:PRC_HICP_AIND(1.0),06/02/26 23:00:00,Annual,Core inflation differential vis-à-vis the euro...,"Overall index excluding energy, food, alcohol ...",Austria,2002,-0.4,NaN,NaN
1,ESTAT:PRC_HICP_AIND(1.0),06/02/26 23:00:00,Annual,Core inflation differential vis-à-vis the euro...,"Overall index excluding energy, food, alcohol ...",Austria,2003,-0.6,NaN,NaN
2,ESTAT:PRC_HICP_AIND(1.0),06/02/26 23:00:00,Annual,Core inflation differential vis-à-vis the euro...,"Overall index excluding energy, food, alcohol ...",Austria,2004,-0.3,NaN,NaN
3,ESTAT:PRC_HICP_AIND(1.0),06/02/26 23:00:00,Annual,Core inflation differential vis-à-vis the euro...,"Overall index excluding energy, food, alcohol ...",Austria,2005,-0.1,NaN,NaN
4,ESTAT:PRC_HICP_AIND(1.0),06/02/26 23:00:00,Annual,Core inflation differential vis-à-vis the euro...,"Overall index excluding energy, food, alcohol ...",Austria,2006,-0.2,NaN,NaN


In [8]:
UNIT = "Annual average index"
COICOP = "All-items HICP"

df_hicp_filtered = df_hicp[(df_hicp["unit"] == UNIT) & (df_hicp["coicop"] == COICOP)].copy()
df_hicp_filtered["TIME_PERIOD"] = df_hicp_filtered["TIME_PERIOD"].astype(int)
df_hicp_filtered = df_hicp_filtered[["geo", "TIME_PERIOD", "OBS_VALUE"]].rename(columns={"OBS_VALUE": "HICP"})
print(df_hicp_filtered.shape)
df_hicp_filtered.head()

(1229, 3)


,geo,TIME_PERIOD,HICP
5006,Albania,2016,101.51
5007,Albania,2017,104.76
5008,Albania,2018,106.59
5009,Albania,2019,108.39
5010,Albania,2020,110.74


In [15]:
# Standardize names across shapefile definitions
df_hicp_filtered["geo"] = df_hicp_filtered["geo"].replace({
    "Türkiye": "Turkey",
    "The Republic of North Macedonia": "North Macedonia",
    "Czechia": "Czech Republic",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Bosnia and Herz.": "Bosnia and Herzegovina",
    "Russian Federation": "Russia",
    "Moldova, Republic of": "Moldova"
})

europe_list = [
    "Albania", "Andorra", "Austria", "Belgium", "Bosnia and Herzegovina", "Bulgaria",
    "Croatia", "Cyprus", "Czech Republic", "Denmark", "Estonia", "Finland", "France", "Germany",
    "Greece", "Hungary", "Iceland", "Ireland", "Italy", "Kosovo", "Latvia", "Liechtenstein",
    "Lithuania", "Luxembourg", "Malta", "Moldova", "Monaco", "Montenegro", "Netherlands",
    "North Macedonia", "Norway", "Poland", "Portugal", "Romania", "San Marino", "Serbia",
    "Slovakia", "Slovenia", "Spain", "Sweden", "Switzerland", "Turkey", "Ukraine", "United Kingdom",
    "Vatican"
]

time_list = list(range(2013, 2024))

df_hicp_filtered = df_hicp_filtered[df_hicp_filtered["geo"].isin(europe_list)].copy()
df_hicp_filtered = df_hicp_filtered[df_hicp_filtered["TIME_PERIOD"].isin(time_list)].copy()

print(df_hicp_filtered.shape)
df_hicp_filtered.head()

(387, 3)


,geo,TIME_PERIOD,HICP
5006,Albania,2016,101.51
5007,Albania,2017,104.76
5008,Albania,2018,106.59
5009,Albania,2019,108.39
5010,Albania,2020,110.74


## 3. Country name normalization

The `geo` field in `df_hicp` uses the official Eurostat names, which in a few cases differ slightly from the ones used in `df_mobility`
(e.g. `Czech Republic` vs `Czechia`, `Turkey` vs `Türkiye`). We map these differences to maximize the number of matches.

Non-European countries present in `df_mobility` (e.g. Australia, Canada, China, South Korea, United Arab Emirates, Vietnam, Cambodia, Palestine, Ukraine) are not covered by the Eurostat HICP dataset: for these the resulting value will correctly be `NaN`.

**Note on the UK**: in the Eurostat file, the United Kingdom's time series stops in 2019 (following Brexit). Mobilities with `Sending/Receiving Country = "United Kingdom"` and `Academic Year >= 2020` will therefore legitimately have `NaN`, not because of a join error.

In [ ]:
# map to align df_mobility country names with the ones used in df_hicp (the 'geo' column)
COUNTRY_NAME_MAP = {
    "Türkiye": "Turkey",
    "The Republic of North Macedonia": "North Macedonia",
    "Czechia": "Czech Republic",
    "Cote d'Ivoire": "Côte d'Ivoire",
    "Bosnia and Herz.": "Bosnia and Herzegovina",
    "Russian Federation": "Russia",
    "Moldova, Republic of": "Moldova"
}

def normalize_country_name(name):
    if pd.isna(name):
        return np.nan
    return COUNTRY_NAME_MAP.get(name, name)

## 4. Building the final dataframe with the join

In [32]:
df_mobility["_sending_country_name"] = df_mobility["Sending Country"].apply(normalize_country_name)
df_mobility["_receiving_country_name"] = df_mobility["Receiving Country"].apply(normalize_country_name)

# join for the sending country, on the academic year
hicp_sending = df_hicp_filtered.rename(
    columns={"geo": "_sending_country_name", "TIME_PERIOD": "Academic Year", "HICP": "Sending Country HICP"}
)
df_mobility = df_mobility.merge(hicp_sending, on=["_sending_country_name", "Academic Year"], how="left")

# join for the receiving country, on the academic year
hicp_receiving = df_hicp_filtered.rename(
    columns={"geo": "_receiving_country_name", "TIME_PERIOD": "Academic Year", "HICP": "Receiving Country HICP"}
)
df_mobility = df_mobility.merge(hicp_receiving, on=["_receiving_country_name", "Academic Year"], how="left")

# drop the helper columns used only for the join
df_mobility = df_mobility.drop(columns=["_sending_country_name", "_receiving_country_name"])

df_mobility.head()

,Academic Year,Mobility Duration,Field of Education,Participant Country,Education Level,Participant Gender,Fewer Opportunities,Participant Age,Sending Country,Sending City,Sending Organization,Receiving Country,Receiving City,Receiving Organization,Sending Country HICP,Receiving Country HICP
0,2023,61,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,Austria,WIEN,WU,Germany,Munich,-,130.4,125.9
1,2023,101,Economics,Germany,ISCED-7 - Second cycle / Master’s or equivalen...,Male,0,27,Austria,WIEN,WU,France,Toulouse,Université Toulouse Capitole,130.4,120.5
2,2023,121,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,Austria,WIEN,WU,France,Paris,-,130.4,120.5
3,2023,171,Business and administration,Germany,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,22,Austria,WIEN,WU,France,Paris,-,130.4,120.5
4,2023,86,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,23,Austria,WIEN,WU,Germany,Hamburg,-,130.4,125.9


In [41]:
# reorder columns to insert the new features right after Sending Country / Receiving Country
new_column_order = []
for col in df_mobility.columns:
    new_column_order.append(col)
    if col == "Sending Country":
        new_column_order.append("Sending Country HICP")
    elif col == "Receiving Country":
        new_column_order.append("Receiving Country HICP")

df_mobility_hicp = df_mobility[new_column_order].loc[:, ~df_mobility[new_column_order].columns.duplicated()]
print(df_mobility_hicp.shape)
df_mobility_hicp.head()

(3172958, 16)


,Academic Year,Mobility Duration,Field of Education,Participant Country,Education Level,Participant Gender,Fewer Opportunities,Participant Age,Sending Country,Sending Country HICP,Sending City,Sending Organization,Receiving Country,Receiving Country HICP,Receiving City,Receiving Organization
0,2023,61,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,Austria,130.4,WIEN,WU,Germany,125.9,Munich,-
1,2023,101,Economics,Germany,ISCED-7 - Second cycle / Master’s or equivalen...,Male,0,27,Austria,130.4,WIEN,WU,France,120.5,Toulouse,Université Toulouse Capitole
2,2023,121,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Female,0,22,Austria,130.4,WIEN,WU,France,120.5,Paris,-
3,2023,171,Business and administration,Germany,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,22,Austria,130.4,WIEN,WU,France,120.5,Paris,-
4,2023,86,Business and administration,Austria,ISCED-6 - First cycle / Bachelor’s or equivale...,Male,0,23,Austria,130.4,WIEN,WU,Germany,125.9,Hamburg,-


## 5. Saving the result

In [42]:
df_mobility_hicp.to_csv("Datasets/Processed/df_mobility_hicp.csv", index=False)
print("Saved df_mobility_hicp.csv")

Saved df_mobility_hicp.csv
